# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [13]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    # TODO: Handle empty input
    if len(argv) == 0:
        return "Error: Please provide a command. Usage: greet <name> or add <x> <y>"

    command = argv[0]

    # TODO: Support: greet <name>
    if command == "greet":
        if len(argv) == 2:
            name = argv[1]
            return f"Hello, {name}!"
        else:
            return "Usage: greet <name>"

    # TODO: Support: add <x> <y>
    elif command == "add":
        if len(argv) == 3:

            # TODO: Use try/except for numeric conversion
            try:
                x = float(argv[1])
                y = float(argv[2])
                result = x + y
                return str(result)

            except ValueError:
                return "Error: Please enter valid numeric input."

        else:
            return "Usage: add <x> <y>"

    # TODO: Return an error/usage message for bad commands
    else:
        return "Error: Invalid command. Usage: greet <name> or add <x> <y>"


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input

Hello, Alice!
7.0
Error: Please enter valid numeric input.


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0.0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [16]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    # TODO: Create ArgumentParser
    parser = argparse.ArgumentParser(description="Filter expenses")

    # TODO: Add optional --category argument
    parser.add_argument("--category", type=str, default=None)

    # TODO: Add optional --min argument (type=float, default=0.0)
    parser.add_argument("--min", type=float, default=0.0)

    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    # TODO: Loop through expenses and keep only matching rows
    filtered = []

    for expense in expenses:
        if category is not None and expense["category"].lower() != category.lower():
            continue

        if expense["amount"] < min_amount:
            continue

        filtered.append(expense)

    return filtered

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    # TODO: Handle empty list
    if not expenses:
        return "No expenses found."

    # TODO: Build a list of output lines
    lines = []

    for expense in expenses:
        line = (
            f'{expense["date"]} | '
            f'{expense["category"]} | '
            f'€{expense["amount"]:.2f} | '
            f'{expense["note"]}'
        )
        lines.append(line)

    # TODO: Return one multi-line string
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))

2026-10-02 | Food | €34.20 | Groceries


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_data_20261005_224700.json` for `budget_data.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [17]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w", encoding="utf-8") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [19]:
from pathlib import Path
import json
import shutil
from datetime import datetime


def make_backup_name(filename):
    """Return a timestamped backup filename."""
    # Get the original filename
    path = Path(filename)

    # Create timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Return new filename
    return f"{path.stem}_{timestamp}{path.suffix}"


def backup_file(source_path, backup_dir="backups"):
    """Create a timestamped backup of a file."""
    # Convert source path to Path object
    source_path = Path(source_path)

    # Validate that source file exists
    if not source_path.is_file():
        return None

    # Create backup directory if needed
    backup_dir = Path(backup_dir)
    backup_dir.mkdir(parents=True, exist_ok=True)

    # Generate backup filename
    backup_name = make_backup_name(source_path.name)

    # Create destination path
    destination_path = backup_dir / backup_name

    # Copy source file to backup folder
    shutil.copy2(source_path, destination_path)

    # Return destination path as string
    return str(destination_path)


# --- TEST ---
backup_path = backup_file("budget_data.json")

if backup_path:
    print("Backup created:", backup_path)
else:
    print("Source file not found.")

Backup created: backups\budget_data_20261109_105529.json


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [20]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [21]:
import csv


def load_expenses_csv(filename):
    """Load expenses from a CSV file and skip malformed rows."""
    expenses = []

    with open(filename, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:

            # Skip rows with an empty category
            if not row["category"] or not row["category"].strip():
                continue

            # Skip rows with a missing or invalid amount
            try:
                amount = float(row["amount"])
            except (ValueError, TypeError):
                continue

            # Create expense dictionary
            expense = {
                "date": row["date"],
                "category": row["category"],
                "amount": amount,
                "note": row["note"]
            }

            expenses.append(expense)

    return expenses


def summarize_by_category(expenses):
    """Return total expenses grouped by category."""
    summary = {}

    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]

        if category not in summary:
            summary[category] = 0

        summary[category] += amount

    return summary


def print_summary(summary):
    """Print a formatted expense summary."""
    print("Expense Summary")
    print("--------------------")

    if not summary:
        print("No expenses found.")
        return

    for category, total in summary.items():
        print(f"{category}: €{total:.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")

summary = summarize_by_category(expenses)

print_summary(summary)

Expense Summary
--------------------
Food: €39.80
Transport: €2.80


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`

   (The exercise uses plain `python` to keep the strings short; on a real machine use the full path to your project's interpreter, as shown in the slides.)
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [22]:
def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    # TODO: Validate minute (0-59) and hour (0-23)
    if not isinstance(minute, int) or not 0 <= minute <= 59:
        raise ValueError("Minute must be between 0 and 59")

    if not isinstance(hour, int) or not 0 <= hour <= 23:
        raise ValueError("Hour must be between 0 and 23")

    if not script_path:
        raise ValueError("Script path cannot be empty")

    # TODO: Return the cron line as a string
    return f"{minute} {hour} * * * python {script_path}"


def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # TODO: Validate HH:MM format
    if not script_path:
        raise ValueError("Script path cannot be empty")

    try:
        hour, minute = map(int, time_str.split(":"))
    except ValueError:
        raise ValueError("Time must be in HH:MM format")

    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError("Invalid time")

    # TODO: Return a command like:
    # schtasks /Create /SC DAILY /TN BudgetBackup /TR "python script.py" /ST 18:30
    return f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "python {script_path}" /ST {time_str}'


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))

print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))

30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN BudgetBackup /TR "python C:/scripts/backup.py" /ST 18:30


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument. It comes **before** the subcommand: `["expenses_suite.json", "list"]`.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [23]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w", encoding="utf-8") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [24]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    # TODO: Handle FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, "r", encoding="utf-8") as file:
            expenses = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

    # TODO: Ensure the top-level object is a list
    if not isinstance(expenses, list):
        return []

    # TODO: Return [] on error
    return [expense for expense in expenses if isinstance(expense, dict)]

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    # TODO: Create parser
    parser = argparse.ArgumentParser(description="Expense Automation Suite")

    # TODO: Add a required filename argument
    parser.add_argument("filename")

    # TODO: Add subcommands: list, summary, backup
    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser("list")
    subparsers.add_parser("summary")
    subparsers.add_parser("backup")

    return parser

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    # TODO: Parse args
    parser = build_suite_parser()
    args = parser.parse_args(argv)

    # TODO: Load data when needed
    if args.command == "backup":
        from pathlib import Path
        from datetime import datetime
        import shutil

        source_path = Path(args.filename)

        if not source_path.is_file():
            return "Error: Source file not found."

        backup_dir = Path("backups")
        backup_dir.mkdir(parents=True, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_name = f"{source_path.stem}_{timestamp}{source_path.suffix}"
        destination_path = backup_dir / backup_name

        shutil.copy2(source_path, destination_path)

        return f"Backup created: {destination_path}"

    expenses = load_json_expenses(args.filename)

    # TODO: Route to list / summary / backup
    if args.command == "list":
        if not expenses:
            return "No expenses found."

        lines = []
        for expense in expenses:
            lines.append(
                f'{expense.get("date", "N/A")} | '
                f'{expense.get("category", "N/A")} | '
                f'€{float(expense.get("amount", 0)):.2f} | '
                f'{expense.get("note", "")}'
            )

        return "\n".join(lines)

    elif args.command == "summary":
        summary = {}

        for expense in expenses:
            category = expense.get("category")
            try:
                amount = float(expense.get("amount", 0))
            except (ValueError, TypeError):
                continue

            if category:
                summary[category] = summary.get(category, 0) + amount

        if not summary:
            return "No expenses found."

        lines = ["Expense Summary", "--------------------"]
        for category, total in summary.items():
            lines.append(f"{category}: €{total:.2f}")

        return "\n".join(lines)


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))

2026-10-01 | Food | €18.50 | Lunch
2026-10-02 | Transport | €2.80 | Metro
2026-10-03 | Food | €21.30 | Dinner
2026-10-04 | Books | €45.00 | Python book

Expense Summary
--------------------
Food: €39.80
Transport: €2.80
Books: €45.00

Backup created: backups\expenses_suite_20261109_105858.json
